# Alternative 3 — GCN sur la topologie de la main

**Objectif :** Exploiter explicitement la structure anatomique de la main : les 21 landmarks sont des nœuds d'un graphe dont les arêtes suivent les os (poignet, doigts, phalanges, paume).

**GCN** (Kipf & Welling) : convolution de graphe avec adjacence normalisée
symétriquement D^-1/2 A D^-1/2, 2 blocs résiduels à 64 dimensions.
Les deux mains partagent les poids ; le pooling (mean + max, masqué pour la main absente)
concatène les deux mains avant la tête de classification 57 classes.

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → jeux d'entraînement et de validation/test strictement disjoints.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit graphe de 21 nœuds × 2 mains, features par nœud : (x, y, z).


In [2]:
dataset = load_dataset()
X_train = torch.from_numpy(dataset["train"]["img_landmarks"]).float()
y_train = torch.from_numpy(dataset["train"]["labels"])
X_val   = torch.from_numpy(dataset["val"]["img_landmarks"]).float()
y_val   = torch.from_numpy(dataset["val"]["labels"])
X_test  = torch.from_numpy(dataset["test"]["img_landmarks"]).float()
y_test  = torch.from_numpy(dataset["test"]["labels"])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 2, 21, 3) | val: (767, 2, 21, 3) | test: (767, 2, 21, 3)


## 2. Architecture

GCN : 21 nœuds, arêtes = connexions anatomiques MediaPipe (23 arêtes + self-loop),
adjacence normalisée symétrique, 2 couches (64D) avec résidus + BatchNorm, pooling mean+max
par main (masqué), tête 128 → 57. ~45k paramètres.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = HandGCN(num_classes=57, hidden=64, num_layers=2)
print(f"params: {count_params(model):,}")


params: 45,049


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 100 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=100, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "gcn_curves.png",
                     title="Alternative 3 — GCN sur la topologie de la main")
print("best val acc:", trainer.best_val_acc)


  ep 1/100 train 3.2330/0.2215 val 2.8469/0.3664 (1.0s) best 0.3664


  ep 2/100 train 2.5460/0.4210 val 2.3589/0.4980 (1.9s) best 0.4980


  ep 3/100 train 2.2237/0.5161 val 2.1612/0.5528 (1.9s) best 0.5528


  ep 4/100 train 2.0406/0.5729 val 2.0838/0.5554 (1.9s) best 0.5554


  ep 5/100 train 1.9279/0.6115 val 2.0118/0.5828 (1.9s) best 0.5828


...

  ep 50/100 train 1.2712/0.8617 val 1.7084/0.6675 (1.9s) best 0.7040


  ep 51/100 train 1.2762/0.8566 val 1.7046/0.6754 (2.0s) best 0.7040


  ep 52/100 train 1.2740/0.8583 val 1.6961/0.6910 (1.9s) best 0.7040


  ep 53/100 train 1.2696/0.8595 val 1.6906/0.6806 (1.1s) best 0.7040
  early stop @ 53 (best val acc 0.7040)


best val acc: 0.7040417209908736


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "gcn_cm.png",
               title="Alternative 3 — GCN sur la topologie de la main — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_gcn.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_gcn.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.7392
macro F1: 0.7166 | weighted F1: 0.7479
macro precision: 0.8094 | macro recall: 0.7146


     class  precision    recall        f1  support
10  mar7ba   0.000000  0.000000  0.000000        3
54   metro   0.000000  0.000000  0.000000        0
52  karhba   0.153846  1.000000  0.266667        2
8    labes   0.500000  0.250000  0.333333        4
32   3ayla   1.000000  0.230769  0.375000       13
17  siye7a   0.280000  0.777778  0.411765        9
39   jadda   0.285714  0.750000  0.413793       16
2    assam   1.000000  0.285714  0.444444        7


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=512):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(raw), batch_size):
            x = torch.from_numpy(raw[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.6639 (drop=+0.0754)
  noise_0.02: acc=0.6258 (drop=+0.1134)
  noise_0.05: acc=0.4089 (drop=+0.3304)
  rotation_5: acc=0.7379 (drop=+0.0013)
  rotation_10: acc=0.7027 (drop=+0.0365)


  rotation_20: acc=0.5867 (drop=+0.1525)
  translation_0.05: acc=0.6409 (drop=+0.0983)
  translation_0.10: acc=0.5434 (drop=+0.1958)
  scale_0.90: acc=0.6023 (drop=+0.1369)
  scale_1.10: acc=0.8162 (drop=-0.0769)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "gcn")
fp16 = ex.quantize_fp16(onnx_path, "gcn")
int8 = ex.quantize_int8(onnx_path, "gcn")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("gcn", metrics, extra=extra)

torch_path = ex.save_torch(model, "gcn")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.392 ms/sample


  fp32: size=0.18 MB latency=0.065 ms acc=0.7392
  fp16: size=0.09 MB latency=0.064 ms acc=0.7379
  int8: size=0.06 MB latency=0.068 ms acc=0.7262
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_gcn.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\gcn.pth (0.18 MB)


## 7. Conclusions

Le GCN force le modèle à n'agréger l'information qu'à travers la structure osseuse
de la main — un a priori anatomique fort. S'il s'avère au moins aussi bon que le MLP avec
~20x moins de paramètres, c'est un candidat sérieux pour le mobile.
